# Maze RL — speed benchmark (instructor only)

Run this once on a **CPU** runtime and once on a **T4 GPU** runtime (`Runtime → Change runtime type`).
It takes about 2 minutes. Copy the printed summary back so the class time budgets can be set from real numbers.

In [ ]:
!pip install -q git+https://github.com/anthoak13/HNR-MazeSolver@maze-rl

In [ ]:
import time, platform, torch
import mazerl
from mazerl.maze import MazeBank
from mazerl.env import MazeEnv
from mazerl.a2c import train

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else platform.processor())
print("torch threads:", torch.get_num_threads())

In [ ]:
# 1. Environment alone (random moves)
bank = MazeBank.generate(500, seed=0)
env = MazeEnv(bank, 512, device=device)
for _ in range(5):
    env.step(torch.randint(0, 4, (512,), device=device)); env.observe()
if device == "cuda": torch.cuda.synchronize()
t = time.time()
for _ in range(100):
    env.step(torch.randint(0, 4, (512,), device=device)); env.observe()
if device == "cuda": torch.cuda.synchronize()
env_rate = 512 * 100 / (time.time() - t)
print(f"environment: {env_rate:,.0f} moves/s")

In [ ]:
# 2. Training loop (3,000 attempts)
run = train(episodes=3000, layouts=2000, device=device, progress=True)
train_secs = run.seconds

In [ ]:
h = run.history
attempts = h["episodes"][-1]
avg_steps = sum(h["steps"]) / len(h["steps"])
moves_per_s = attempts * avg_steps / train_secs
print(f"training: {attempts} attempts in {train_secs:.0f}s  ->  {moves_per_s:,.0f} moves/s, {attempts / train_secs:.0f} attempts/s")
for steps in (100, 60, 40):
    full = 126_000 * steps / moves_per_s
    print(f"  full 126k-attempt run at {steps} moves/attempt: {full / 60:.0f} min")
print(f"  attempts in a 5-minute class run (early training, ~100 moves each): {300 * moves_per_s / 100:,.0f}")
print("SUMMARY", dict(device=device, env_rate=round(env_rate), moves_per_s=round(moves_per_s)))